# Train the emergency-vehicle detector on Kaggle (no extra account needed)

Fine-tunes YOLOv8s on the Kaggle dataset [EV vs NEV Annotated Vehicle Image Dataset](https://www.kaggle.com/datasets/rohitshitole96/ev-vs-nev-annotated-vehicle-image-dataset) (CC BY-SA 4.0): traffic images with boxes for **emergency vehicles (EV)** and **non-emergency vehicles (NEV)**, already in YOLO format.

**Kaggle settings:** Add Input → the dataset above; Accelerator → **GPU T4**; Internet → **On**.

The classes are renamed to `emergency vehicle` and `vehicle`, so the app recognises the emergency class by name (`traffic/detection.py` looks for "emergency", "ambulance", "fire" or "police").

In [ ]:
!pip install -q ultralytics
import glob, json, os, yaml
roots = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert roots, "Add the EV vs NEV dataset as Input first."
src_yaml = roots[0]
base = os.path.dirname(src_yaml)
original = yaml.safe_load(open(src_yaml))
print("original data.yaml:", original)
for split in ("train", "val"):
    print(split, len(glob.glob(f"{base}/{split}/images/*")), "images,", len(glob.glob(f"{base}/{split}/labels/*.txt")), "label files")

In [ ]:
# New data.yaml with absolute paths and clearer class names
names = original["names"]
names = list(names.values()) if isinstance(names, dict) else list(names)
rename = {"EV": "emergency vehicle", "NEV": "vehicle"}
new_names = [rename.get(str(n).strip().upper(), str(n)) for n in names]
data = {"train": f"{base}/train/images", "val": f"{base}/val/images", "nc": len(new_names), "names": new_names}
yaml.safe_dump(data, open("/kaggle/working/data.yaml", "w"))
print(data)

In [ ]:
# How many boxes of each class?
from collections import Counter
for split in ("train", "val"):
    counts = Counter()
    for f in glob.glob(f"{base}/{split}/labels/*.txt"):
        for line in open(f):
            if line.strip():
                counts[new_names[int(line.split()[0])]] += 1
    print(split, dict(counts))

## Train
YOLOv8s pretrained on COCO, up to 50 epochs; stops early if validation mAP does not improve for 10 epochs.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
model.train(data="/kaggle/working/data.yaml", epochs=50, imgsz=640, batch=16, patience=10, device=0,
            project="/kaggle/working/runs", name="emergency", exist_ok=True, plots=True, verbose=False)

## Evaluate
The dataset has only train and validation splits, and the validation split was also used to pick the best epoch, so these numbers are slightly optimistic.

In [ ]:
best = YOLO("/kaggle/working/runs/emergency/weights/best.pt")
m = best.val(data="/kaggle/working/data.yaml", split="val", verbose=False)
per_class = {best.names[int(c)]: {"precision": round(float(p), 3), "recall": round(float(r), 3),
                                  "mAP50": round(float(a50), 3), "mAP50_95": round(float(a), 3)}
             for c, p, r, a50, a in zip(m.box.ap_class_index, m.box.p, m.box.r, m.box.ap50, m.box.ap)}
results = {"dataset": "EV vs NEV Annotated Vehicle Image Dataset (Kaggle, CC BY-SA 4.0)", "model": "YOLOv8s",
           "mAP50": round(float(m.box.map50), 3), "mAP50_95": round(float(m.box.map), 3), "per_class": per_class}
json.dump(results, open("/kaggle/working/emergency_metrics.json", "w"), indent=2)
print(json.dumps(results, indent=2))

In [ ]:
from IPython.display import Image, display
for name in ("results.png", "confusion_matrix_normalized.png", "val_batch0_pred.jpg"):
    p = f"/kaggle/working/runs/emergency/{name}"
    if os.path.exists(p):
        display(Image(p, width=900))

In [ ]:
import shutil
shutil.copy("/kaggle/working/runs/emergency/weights/best.pt", "/kaggle/working/emergency_yolov8s.pt")
print("Download emergency_yolov8s.pt from the Output tab and put it in the app's models/ folder.")